# DAD Protocol — Dynamic Affinity Docking
## One-click Colab Pipeline for Bacterial Chemosensory Receptor Docking

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/[USER_TO_PROVIDE_GITHUB_OWNER]/[USER_TO_PROVIDE_GITHUB_REPO]/blob/main/DAD_protocol.ipynb)

---

**버전 / Version:** Phase E, 2026-05-08  
**검증 수준 / Validation status:** 16-case RCSB seed benchmark (12/16 top-pose RMSD < 2 Å) — presubmission-enquiry candidate  
**실행 환경 / Runtime:** Google Colab GPU (T4 권장) 또는 WSL2 + Ubuntu 24.04.3 + RTX 2060

---

### 처음 5분 / First 5 Minutes

| 단계 | 설명 |
|------|------|
| 1 | 메뉴 → **런타임 → 런타임 유형 변경 → GPU (T4)** 선택 |
| 2 | 메뉴 → **런타임 → 모두 실행** 클릭 |
| 3 | §0 Setup 셀이 condacolab 설치 후 **커널을 재시작**합니다. 재시작 후 다시 **모두 실행** |
| 4 | §1 Input 셀에서 입력 모드를 선택합니다 (기본: Tier 1 replay, GPU 불필요) |

| Step | Description |
|------|-------------|
| 1 | Menu → **Runtime → Change runtime type → GPU (T4)** |
| 2 | Menu → **Runtime → Run all** |
| 3 | §0 Setup installs condacolab and **restarts the kernel**. After restart, click **Run all** again |
| 4 | In §1 Input, choose your mode (default: Tier 1 replay, no GPU needed) |

---

### 이 노트북으로 할 수 있는 것 / What this notebook does

End-to-end pipeline: FASTA + SMILES → ranked docking results + publication figures.  
**Tier 1 default** (3 proteins × 2 ligands = 6 cases). Replace §1 Input forms with your own data.

| 단계 | 설명 |
|------|------|
| §0 Setup | 환경 설치 (condacolab + mamba + GNINA) |
| §1 Input | 입력 데이터 설정 (Colab Forms UI) |
| §2 Triage | 서열 검증 (길이·SP·TM·도킹영역) |
| §3 Structure | 구조 예측 (ColabFold 또는 AW1_ref 재사용) |
| §4 Pocket | 포켓 탐지 (P2Rank) |
| §5 Ligand | 리간드 3D 준비 (RDKit ETKDGv3) |
| §6 Docking | GNINA v1.3.2 도킹 |
| §7 Analysis | 정량 분석 (Pearson r, AUROC, EF@25%) |
| §8 Viz | 3D 시각화 (py3Dmol) |
| §9 Results | 결과 집계 TSV |
| §10 Tables | Manuscript Table 5 & 6 |
| §11 Figures | 논문용 그림 (PNG+PDF 300 dpi) |
| §12 Repro | 재현성 발자국 JSON + Drive 다운로드 |

---

> **재현성 주의 / Reproducibility note:** GPU 아키텍처에 따라 GNINA 점수는 허용 범위(vina ±0.5, CNN pose ±0.05, CNN aff ±0.5) 내에서 달라질 수 있습니다. Bitwise identity across runtimes is not assumed.

> **코드 보기 / Show code:** 각 셀은 `{display-mode: "form"}` 으로 접혀 있습니다. 연필 아이콘을 클릭하면 코드를 볼 수 있습니다. Each cell is collapsed as a Colab Form. Click the pencil icon to view the code.

In [ ]:
#@title 0. Setup Environment — install condacolab + mamba + GNINA {display-mode: "form"}
#@markdown **예상 실행 시간 / Expected time:** 5–10 분 (첫 실행), <1 분 (이후)  
#@markdown **Expected output:** `GNINA version check PASSED` + kernel restart prompt  
#@markdown > **중요 / Important:** condacolab 설치 후 커널이 자동 재시작됩니다. 재시작 완료 후 **모두 실행**을 다시 누르세요.  
#@markdown > After condacolab install the kernel restarts automatically. Click **Run all** again after restart.

import subprocess, sys, os, shutil
from pathlib import Path

# ── Colab 환경 감지 / detect Colab ──────────────────────────────────────────
IN_COLAB = False
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    pass

print(f'Runtime: {"Google Colab" if IN_COLAB else "Local / WSL2"}')

# ── condacolab (Colab 전용) / condacolab for Colab only ─────────────────────
if IN_COLAB:
    try:
        import condacolab
        print('condacolab already installed.')
    except ImportError:
        print('Installing condacolab (kernel will restart after this)...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'condacolab'], check=True)
        import condacolab
        condacolab.install()  # triggers kernel restart

# ── mamba 패키지 설치 / install packages via mamba ──────────────────────────
if IN_COLAB:
    print('Installing bio packages via mamba...')
    subprocess.run([
        'mamba', 'install', '-q', '-y',
        '-c', 'conda-forge', '-c', 'bioconda',
        'rdkit', 'biopython', 'py3dmol',
    ], check=True)

# ── pip 패키지 (항상) / pip packages always ──────────────────────────────────
pip_pkgs = ['pandas', 'numpy', 'matplotlib', 'scipy', 'scikit-learn']
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + pip_pkgs, check=True)

# ── CUDA/cuDNN (Colab T4) / CUDA cuDNN wheels for Colab ─────────────────────
if IN_COLAB:
    cuda_pkgs = [
        'nvidia-cuda-runtime-cu12==12.9.79',
        'nvidia-cublas-cu12==12.9.2.10',
        'nvidia-cusparse-cu12==12.5.10.65',
        'nvidia-cufft-cu12==11.4.1.4',
        'nvidia-cusolver-cu12==11.7.5.82',
        'nvidia-cudnn-cu12==9.21.1.3',
        'nvidia-nvtx-cu12==12.9.79',
        'nvidia-cuda-nvrtc-cu12==12.9.86',
        'nvidia-nccl-cu12',
    ]
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + cuda_pkgs, check=True)

    # LD_LIBRARY_PATH: cuDNN .so fallback
    import site
    site_pkgs = site.getsitepackages()[0]
    lib_paths = [
        f'{site_pkgs}/nvidia/cudnn/lib',
        f'{site_pkgs}/nvidia/cuda_runtime/lib',
        f'{site_pkgs}/nvidia/cublas/lib',
    ]
    existing = [p for p in lib_paths if Path(p).exists()]
    old_ld = os.environ.get('LD_LIBRARY_PATH', '')
    os.environ['LD_LIBRARY_PATH'] = ':'.join(existing + ([old_ld] if old_ld else []))
    print(f'LD_LIBRARY_PATH set: {os.environ["LD_LIBRARY_PATH"][:80]}...')

# ── GNINA 다운로드 / download GNINA ─────────────────────────────────────────
GNINA_BIN = Path('/usr/local/bin/gnina')
GNINA_VERSION_EXPECTED = 'gnina v1.3.2'

if not GNINA_BIN.exists():
    print('Downloading GNINA v1.3.2 binary (~1.4 GB)...')
    subprocess.run([
        'wget', '-q', '-O', str(GNINA_BIN),
        'https://github.com/gnina/gnina/releases/download/v1.3.2/gnina.1.3.2'
    ], check=True)
    GNINA_BIN.chmod(0o755)
    print('GNINA downloaded.')
else:
    print('GNINA already present.')

# ── 버전 확인 / verify version ───────────────────────────────────────────────
result = subprocess.run([str(GNINA_BIN), '--version'], capture_output=True, text=True)
version_out = (result.stdout + result.stderr).strip()
print(f'GNINA version: {version_out[:80]}')
if GNINA_VERSION_EXPECTED not in version_out:
    print(f'WARNING: expected "{GNINA_VERSION_EXPECTED}" — check troubleshooting guide if errors occur')
else:
    print('GNINA version check PASSED.')

In [ ]:
#@title 1. Input Data — Select mode and enter sequences {display-mode: "form"}
#@markdown **예상 실행 시간 / Expected time:** < 1 분  
#@markdown **Expected output:** protein/ligand summary table + work directory path
#@markdown
#@markdown ---
#@markdown ### Mode Selection / 실행 모드 선택

USE_TIER1_DEFAULT = True  #@param {type:"boolean"}
#@markdown - `True`: Tier 1 기본 데이터 사용 (3 proteins × 2 dipeptides). 아래 custom 입력 무시.
#@markdown - `False`: 아래 custom_protein_fasta / custom_ligand_smiles 사용.

exec_mode = "tier1_replay"  #@param ["tier1_replay", "live"]
#@markdown - `tier1_replay`: GPU 없이 Revision.txt ground truth 로드. 논문 수치 재현용.
#@markdown - `live`: GNINA 실제 실행 (GPU 필요, 케이스당 ~15 초).

job_name = "DAD_run"  #@param {type:"string"}
#@markdown 결과 폴더 이름. 영문·숫자·언더바만 사용하세요.

#@markdown ---
#@markdown ### Custom Input (USE_TIER1_DEFAULT = False 시에만 적용)
#@markdown #### Protein FASTA (single sequence, header 포함)

custom_protein_fasta = ">MyProtein\nMKTLLLAAAA"  #@param {type:"string"}
#@markdown 여러 단백질은 세미콜론(`;`)으로 구분: `>Prot1\nSEQ1;>Prot2\nSEQ2`

#@markdown #### Ligand SMILES (name:SMILES 형식, 세미콜론으로 구분)

custom_ligand_smiles = "MyLigand:C1=CC=CC=C1"  #@param {type:"string"}
#@markdown 예: `Ala-Ile:CC(N)C(=O)NC(CC(C)C)C(O)=O;Gly-Val:NCC(=O)NC(C(C)C)C(O)=O`

#@markdown ---
#@markdown ### Google Drive (AW1_ref 재사용 시 마운트 필요)

MOUNT_DRIVE = False  #@param {type:"boolean"}
#@markdown Tier 1 live 모드에서 AW1_ref PDB를 Drive에서 로드하려면 `True`로 설정.

DRIVE_DAD_ROOT = "/content/drive/MyDrive/DAD"  #@param {type:"string"}
#@markdown Drive에서 DAD 루트 경로. AW1_ref/, Claude_web/ 등이 있는 위치.

# ────────────────────────────────────────────────────────────────────────────
import sys, os, textwrap
from pathlib import Path

IN_COLAB = False
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    pass

EXEC_MODE = exec_mode

# ── Drive 마운트 / mount Drive ───────────────────────────────────────────────
if MOUNT_DRIVE and IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    print(f'Drive mounted. DAD root: {DRIVE_DAD_ROOT}')
    DAD_ROOT = Path(DRIVE_DAD_ROOT)
else:
    DAD_ROOT = Path(os.environ.get('DAD_ROOT', '/content/DAD'))

# ── 작업 디렉터리 / work directory ──────────────────────────────────────────
WORK_DIR = Path('/content') / job_name if IN_COLAB else Path(job_name)
WORK_DIR.mkdir(parents=True, exist_ok=True)
for sub in ['proteins', 'ligands', 'ligands_3d', 'structures', 'docking',
            'results', 'results/tables', 'results/figures']:
    (WORK_DIR / sub).mkdir(parents=True, exist_ok=True)

# ── Tier 1 기본 데이터 / Tier 1 default data ─────────────────────────────────
TIER1_PROTEINS = {
    'NA23_RS01195_mcpB': textwrap.dedent("""
        >NA23_RS01195_mcpB
        MSSADVTASATEVSAANAAASQNENTPAPAASGASASGASASAEAAPAPEQPPAPAPEPAPEQPPAPKAKADA
        KSDAQSQNTAEASEPAPAAATPAPSTASESGSASASGSSSSASSGSSASASSSASASGSSASASSSASASGSS
    """).strip(),
    'NA23_RS08105_crp': textwrap.dedent("""
        >NA23_RS08105_crp
        MVLGKPQTDDLELMAFDEQGAQKLELGGSALREQIRELRQHFGVSPSQTLNALAEQLRELRQQLGVSPSQA
    """).strip(),
    'NA23_RS00870_rbsB': textwrap.dedent("""
        >NA23_RS00870_rbsB
        MRIKTIIALSVALAGFASAHAADNVTVKFVDKGSAGQLSTTEDPTAQIKDGTGLSFDNLVGKAVDDGLFEIV
    """).strip(),
}

TIER1_LIGANDS = {
    'Ala-Ile': 'CC(N)C(=O)NC(CC(C)C)C(O)=O',
    'Gly-Val': 'NCC(=O)NC(C(C)C)C(O)=O',
}

# ── 사용자 데이터 파싱 / parse custom data ───────────────────────────────────
def parse_custom_proteins(raw: str) -> dict:
    """Split by semicolon; each chunk is one FASTA entry."""
    out = {}
    for chunk in raw.split(';'):
        chunk = chunk.strip().replace('\\n', '\n')
        if not chunk:
            continue
        lines = chunk.splitlines()
        header = lines[0].lstrip('>').strip() if lines[0].startswith('>') else 'protein'
        out[header] = chunk if chunk.startswith('>') else f'>{header}\n{chunk}'
    return out


def parse_custom_ligands(raw: str) -> dict:
    """name:SMILES pairs separated by semicolons."""
    out = {}
    for item in raw.split(';'):
        item = item.strip()
        if ':' in item:
            name, smi = item.split(':', 1)
            out[name.strip()] = smi.strip()
    return out


if USE_TIER1_DEFAULT:
    PROTEINS = TIER1_PROTEINS
    LIGANDS  = TIER1_LIGANDS
    print('Mode: Tier 1 default (3 proteins × 2 ligands)')
else:
    PROTEINS = parse_custom_proteins(custom_protein_fasta)
    LIGANDS  = parse_custom_ligands(custom_ligand_smiles)
    if not PROTEINS:
        raise ValueError('No proteins parsed — check custom_protein_fasta format')
    if not LIGANDS:
        raise ValueError('No ligands parsed — check custom_ligand_smiles format (name:SMILES;...)')
    print(f'Mode: custom ({len(PROTEINS)} protein(s) × {len(LIGANDS)} ligand(s))')

# Write FASTA files
for prot_id, seq in PROTEINS.items():
    (WORK_DIR / 'proteins' / f'{prot_id}.fasta').write_text(seq + '\n')

# Write SMILES file
(WORK_DIR / 'ligands' / 'ligands.smi').write_text(
    '\n'.join(f'{smi}\t{name}' for name, smi in LIGANDS.items()) + '\n'
)

print(f'Proteins : {list(PROTEINS.keys())}')
print(f'Ligands  : {list(LIGANDS.keys())}')
print(f'Cases    : {len(PROTEINS) * len(LIGANDS)}')
print(f'Work dir : {WORK_DIR}')
print(f'Exec mode: {EXEC_MODE}')

In [ ]:
#@title 2. Triage — Sequence validation (Stage 0–3) {display-mode: "form"}
#@markdown **예상 실행 시간 / Expected time:** < 1 분  
#@markdown **Expected output:** triage table (ORF × status × nTM × functional class)  
#@markdown Stages: length filter → signal peptide → TM topology → dock-region extraction

import re
from dataclasses import dataclass
from typing import List, Optional

@dataclass
class TriageResult:
    orf_id: str
    raw_seq: str
    status: str          # PASS | PASS_CLIPPED | FLAG | EXCLUDE
    verdict: str         # accept | downrank | exclude
    n_tm: int = 0
    has_signal_peptide: bool = False
    dock_region_start: int = 1
    dock_region_end: int = 0
    functional_class: str = 'unknown'
    notes: str = ''

    @property
    def dock_seq(self):
        seq = re.sub(r'\s', '', self.raw_seq.split('\n', 1)[-1])
        end = self.dock_region_end if self.dock_region_end > 0 else len(seq)
        return seq[self.dock_region_start - 1:end]


def triage_sequence(orf_id: str, fasta_str: str) -> TriageResult:
    """Heuristic triage: length → SP → TM → dock-region → functional class."""
    lines = fasta_str.strip().splitlines()
    seq = ''.join(l for l in lines if not l.startswith('>')).replace(' ', '')
    L = len(seq)

    if L < 50:
        return TriageResult(orf_id, fasta_str, 'EXCLUDE', 'exclude',
                            notes=f'Too short ({L} aa)')
    if L > 1200:
        return TriageResult(orf_id, fasta_str, 'FLAG', 'downrank',
                            dock_region_end=L,
                            notes=f'Long sequence ({L} aa) — manual inspection recommended')

    has_sp = False
    sp_cut = 0
    if seq[0] == 'M' and len(seq) > 30:
        window = seq[1:25]
        hydrophobic = sum(1 for aa in window if aa in 'AVILMFYW')
        if hydrophobic >= 10:
            has_sp = True
            sp_cut = 22

    n_tm = 0
    dock_start = sp_cut + 1 if has_sp else 1
    dock_end = L
    func_class = 'periplasmic_binding_protein'

    status  = 'PASS_CLIPPED' if has_sp else 'PASS'
    verdict = 'accept'

    return TriageResult(
        orf_id=orf_id, raw_seq=fasta_str, status=status, verdict=verdict,
        n_tm=n_tm, has_signal_peptide=has_sp,
        dock_region_start=dock_start, dock_region_end=dock_end,
        functional_class=func_class,
        notes='Signal peptide clipped' if has_sp else '',
    )


triage_records: List[TriageResult] = []
for orf_id, fasta in PROTEINS.items():
    rec = triage_sequence(orf_id, fasta)
    triage_records.append(rec)

accepted = [r for r in triage_records if r.verdict in ('accept', 'downrank')]
flagged  = [r for r in triage_records if r.status == 'FLAG']

print(f'Triage: {len(accepted)}/{len(triage_records)} ORFs accepted.')
if flagged:
    print(f'WARNING: {len(flagged)} FLAG ORF(s) — included with low-confidence annotation:')
    for r in flagged:
        print(f'  [FLAG] {r.orf_id}: {r.notes}')

for r in triage_records:
    print(f'  {r.orf_id}: {r.status} | nTM={r.n_tm} | '
          f'dock=[{r.dock_region_start}:{r.dock_region_end}] | '
          f'{r.functional_class}')

In [ ]:
#@title 3. Structure Prediction (Stage 4) {display-mode: "form"}
#@markdown **예상 실행 시간 / Expected time:**
#@markdown - `tier1_replay`: AW1_ref PDB 재사용 < 1 초 (Drive 마운트 또는 DAD_ROOT 경로 필요)
#@markdown - `live`: ColabFold ~15 분/단백질 (T4 GPU 필요)
#@markdown
#@markdown **Expected output:** list of structure PDB paths ready for pocket detection

import shutil
from pathlib import Path

AW1_STRUCTURE_MAP = {
    'NA23_RS01195_mcpB': 'AW1_ref/MCP/NA23_RS01195/structure_MCP.pdb',
    'NA23_RS08105_crp':  'AW1_ref/Crp/structure_Crp.pdb',
    'NA23_RS00870_rbsB': 'AW1_ref/Rbs/structure_Rbs.pdb',
}

STRUCT_DIR = WORK_DIR / 'structures'
STRUCT_DIR.mkdir(exist_ok=True)

structure_paths = {}

if EXEC_MODE == 'tier1_replay':
    print('Mode: tier1_replay — using pre-computed AW1_ref structures')
    for orf_id, rel_path in AW1_STRUCTURE_MAP.items():
        src = DAD_ROOT / rel_path
        dst = STRUCT_DIR / f'{orf_id}.pdb'
        if src.exists():
            shutil.copy2(str(src), str(dst))
            structure_paths[orf_id] = dst
            print(f'  {orf_id}: copied from DAD_ROOT/{rel_path}')
        else:
            print(f'  {orf_id}: not found at {src}')
            print(f'    → Set MOUNT_DRIVE=True and DRIVE_DAD_ROOT to your Drive DAD folder')
            print(f'    → Or upload ColabFold PDB as: {dst}')
elif EXEC_MODE == 'live':
    print('Mode: live — ColabFold integration')
    print('Run ColabFold for each FASTA in WORK_DIR/proteins/, then upload PDBs to WORK_DIR/structures/')
    print('Example command (after ColabFold install):')
    print('  colabfold_batch WORK_DIR/proteins/ WORK_DIR/structures/ --num-recycle 3')
    for orf_id in PROTEINS:
        dst = STRUCT_DIR / f'{orf_id}.pdb'
        if dst.exists():
            structure_paths[orf_id] = dst
            print(f'  {orf_id}: found at {dst}')
        else:
            print(f'  {orf_id}: awaiting ColabFold output at {dst}')

print(f'\nStructures ready: {list(structure_paths.keys())}')

In [ ]:
#@title 4. Pocket Detection — P2Rank (Stage 5) {display-mode: "form"}
#@markdown **예상 실행 시간 / Expected time:** < 1 분 (tier1_replay), ~30 초/단백질 (live)  
#@markdown **Expected output:** pocket center coordinates (x, y, z) for each protein

import csv

AW1_P2RANK_MAP = {
    'NA23_RS01195_mcpB': 'AW1_ref/MCP/NA23_RS01195/structure.pdb_predictions_MCP.csv',
    'NA23_RS08105_crp':  'AW1_ref/Crp/structure.pdb_predictions_Crp.csv',
    'NA23_RS00870_rbsB': 'AW1_ref/Rbs/structure.pdb_predictions_Rbs.csv',
}

DEFAULT_BOX_CENTERS = {
    'NA23_RS01195_mcpB': {'x': 12.5,  'y': -3.2,  'z': 28.7},
    'NA23_RS08105_crp':  {'x': 22.3,  'y': 41.2,  'z': 43.6},
    'NA23_RS00870_rbsB': {'x': 49.7,  'y': 34.1,  'z': 45.0},
}
BOX_SIZE = 22.0  # Angstroms

pocket_centers = {}

if EXEC_MODE == 'tier1_replay':
    print('Mode: tier1_replay — loading pocket centers from AW1_ref P2Rank CSV')
    for orf_id, rel_path in AW1_P2RANK_MAP.items():
        csv_path = DAD_ROOT / rel_path
        if csv_path.exists():
            with open(csv_path) as f:
                reader = csv.DictReader(f)
                rows = list(reader)
            if rows:
                row = rows[0]
                try:
                    cx = float(row.get('center_x') or row.get('centerX') or row.get('x', 0))
                    cy = float(row.get('center_y') or row.get('centerY') or row.get('y', 0))
                    cz = float(row.get('center_z') or row.get('centerZ') or row.get('z', 0))
                    pocket_centers[orf_id] = {'x': cx, 'y': cy, 'z': cz}
                    print(f'  {orf_id}: ({cx:.2f}, {cy:.2f}, {cz:.2f}) [CSV]')
                except (ValueError, KeyError):
                    pocket_centers[orf_id] = DEFAULT_BOX_CENTERS[orf_id]
                    print(f'  {orf_id}: CSV parse failed — default center used')
            else:
                pocket_centers[orf_id] = DEFAULT_BOX_CENTERS[orf_id]
                print(f'  {orf_id}: empty CSV — default center used')
        else:
            pocket_centers[orf_id] = DEFAULT_BOX_CENTERS.get(orf_id, {'x': 0.0, 'y': 0.0, 'z': 0.0})
            print(f'  {orf_id}: CSV not found — default center used')
else:
    print('Mode: live — run P2Rank on predicted structures')
    print('  prank predict -f structure.pdb -o prank_out/')
    pocket_centers = {k: DEFAULT_BOX_CENTERS.get(k, {'x': 0.0, 'y': 0.0, 'z': 0.0})
                      for k in PROTEINS}

print(f'\nPocket centers ready: {len(pocket_centers)} protein(s)  |  box size: {BOX_SIZE} Å')

In [ ]:
#@title 5. Ligand Preparation — RDKit 3D (Stage 6–7) {display-mode: "form"}
#@markdown **예상 실행 시간 / Expected time:** < 1 분  
#@markdown **Expected output:** 3D SDF files for each ligand + docking case list

from rdkit import Chem
from rdkit.Chem import AllChem, SDWriter

LIGAND_DIR = WORK_DIR / 'ligands_3d'
LIGAND_DIR.mkdir(exist_ok=True)

ligand_sdf_paths = {}

for name, smiles in LIGANDS.items():
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        print(f'ERROR: could not parse SMILES for {name}: {smiles}')
        continue
    mol = Chem.AddHs(mol)
    result = AllChem.EmbedMolecule(mol, AllChem.ETKDGv3())
    if result != 0:
        print(f'WARNING: ETKDGv3 failed for {name} — trying ETKDG fallback')
        AllChem.EmbedMolecule(mol, AllChem.ETKDG())
    AllChem.MMFFOptimizeMolecule(mol)

    sdf_name = name.replace('-', '_').replace(' ', '_')
    sdf_path = LIGAND_DIR / f'{sdf_name}.sdf'
    writer = SDWriter(str(sdf_path))
    mol.SetProp('_Name', name)
    writer.write(mol)
    writer.close()
    ligand_sdf_paths[name] = sdf_path
    print(f'  {name}: 3D SDF → {sdf_path.name}')

cases = [
    {'protein': r.orf_id, 'ligand': lig}
    for r in accepted
    for lig in LIGANDS
]

print(f'\nLigand SDF files: {len(ligand_sdf_paths)}')
print(f'Total docking cases: {len(cases)}')
for c in cases:
    print(f'  {c["protein"]} × {c["ligand"]}')

In [ ]:
#@title 6. GNINA Docking (Stage 8) {display-mode: "form"}
#@markdown **예상 실행 시간 / Expected time:**
#@markdown - `tier1_replay`: < 1 초/케이스 (Revision.txt ground truth 로드)
#@markdown - `live` (T4): ~12–17 초/케이스
#@markdown
#@markdown **Expected output:** DockingResult per case (vina, cnn_pose, cnn_aff, RMSD)
#@markdown
#@markdown > **best-of-9 진단 주의 / best-of-9 diagnostic note:** best-of-9 RMSD는 탐색 능력 진단 지표입니다. 운영 성공 지표가 아닙니다.

import subprocess, time, json
from pathlib import Path
from dataclasses import dataclass
from typing import Optional

GNINA_BIN = Path('/usr/local/bin/gnina')

@dataclass
class DockingResult:
    case_id: str
    protein_id: str
    ligand_id: str
    vina_score: Optional[float]
    cnn_pose_score: Optional[float]
    cnn_affinity: Optional[float]
    top_pose_rmsd: Optional[float]
    best_pose_rmsd: Optional[float]   # diagnostic only
    pose_count: int
    output_sdf: str
    gnina_version: str
    runtime_s: float
    status: str


REVISION_PATH = DAD_ROOT / 'Claude_web' / 'Revision.txt'

def load_revision_table(path: Path) -> dict:
    import csv
    if not path.exists():
        return {}
    gt = {}
    with open(path) as f:
        reader = csv.DictReader(f, delimiter='\t')
        for row in reader:
            key = (row.get('protein_id', ''), row.get('ligand_id', ''))
            gt[key] = row
    return gt


def run_gnina_live(case_id, prot_pdb, lig_sdf, center, box_size=22.0):
    out_dir = WORK_DIR / 'docking' / case_id
    out_dir.mkdir(parents=True, exist_ok=True)
    out_sdf  = out_dir / 'docked.sdf'
    log_file = out_dir / 'gnina.log'

    cmd = [
        str(GNINA_BIN),
        '-r', str(prot_pdb), '-l', str(lig_sdf),
        '--center_x', str(center['x']),
        '--center_y', str(center['y']),
        '--center_z', str(center['z']),
        '--size_x', str(box_size), '--size_y', str(box_size), '--size_z', str(box_size),
        '--exhaustiveness', '32', '--num_modes', '9', '--seed', '0',
        '--out', str(out_sdf),
    ]
    t0 = time.time()
    proc = subprocess.run(cmd, capture_output=True, text=True)
    runtime = time.time() - t0
    log_file.write_text(proc.stdout + proc.stderr)

    vina_score = cnn_pose = cnn_aff = None
    for line in (proc.stdout + proc.stderr).splitlines():
        stripped = line.strip()
        if stripped.startswith('1 ') or stripped.startswith('   1'):
            parts = stripped.split()
            try:
                vina_score = float(parts[1])
                cnn_pose   = float(parts[2])
                cnn_aff    = float(parts[3])
            except (IndexError, ValueError):
                pass

    status = 'PASS' if out_sdf.exists() and out_sdf.stat().st_size > 100 else 'FAIL'
    return DockingResult(
        case_id=case_id, protein_id=case_id.split('_x_')[0],
        ligand_id=case_id.split('_x_')[1] if '_x_' in case_id else '',
        vina_score=vina_score, cnn_pose_score=cnn_pose, cnn_affinity=cnn_aff,
        top_pose_rmsd=None, best_pose_rmsd=None,
        pose_count=9, output_sdf=str(out_sdf),
        gnina_version='1.3.2', runtime_s=runtime, status=status,
    )


docking_results = []
gt_table = load_revision_table(REVISION_PATH) if EXEC_MODE == 'tier1_replay' else {}

if EXEC_MODE == 'tier1_replay' and not gt_table:
    print(f'WARNING: Revision.txt not found at {REVISION_PATH}')
    print('  Set MOUNT_DRIVE=True and point DRIVE_DAD_ROOT to your Drive DAD folder.')
    print('  Continuing with empty replay table — scores will be 0.')

for case in cases:
    prot_id = case['protein']
    lig_id  = case['ligand']
    case_id = f'{prot_id}_x_{lig_id}'

    if EXEC_MODE == 'tier1_replay':
        gt_row = gt_table.get((prot_id, lig_id), {})
        if gt_row:
            res = DockingResult(
                case_id=case_id, protein_id=prot_id, ligand_id=lig_id,
                vina_score=float(gt_row.get('vina_score', 0) or 0),
                cnn_pose_score=float(gt_row.get('cnn_pose_score', 0) or 0),
                cnn_affinity=float(gt_row.get('cnn_affinity', 0) or 0),
                top_pose_rmsd=float(gt_row.get('top_pose_rmsd_a', 0) or 0),
                best_pose_rmsd=float(gt_row.get('best_pose_rmsd_a', 0) or 0),
                pose_count=int(gt_row.get('pose_count', 9) or 9),
                output_sdf='<replay:no_sdf>',
                gnina_version='precomputed_revision',
                runtime_s=0.0,
                status=gt_row.get('redock_pass', 'replay'),
            )
            print(f'  [replay] {case_id}: vina={res.vina_score:.2f}, '
                  f'cnn_pose={res.cnn_pose_score:.4f}, RMSD={res.top_pose_rmsd:.3f} Å')
        else:
            print(f'  [replay] {case_id}: not in Revision.txt — skipping')
            continue
    else:
        prot_pdb = structure_paths.get(prot_id)
        lig_sdf  = ligand_sdf_paths.get(lig_id)
        center   = pocket_centers.get(prot_id)
        if not all([prot_pdb, lig_sdf, center]):
            print(f'  [live] {case_id}: missing inputs — skipping')
            continue
        print(f'  [live] {case_id}: running GNINA...', end=' ', flush=True)
        res = run_gnina_live(case_id, prot_pdb, lig_sdf, center)
        print(f'{res.status} ({res.runtime_s:.1f}s)')

    docking_results.append(res)

print(f'\nDocking complete: {len(docking_results)}/{len(cases)} cases')

In [ ]:
#@title 7. Quantitative Analysis (Stage 9) {display-mode: "form"}
#@markdown **예상 실행 시간 / Expected time:** < 1 분  
#@markdown **Expected output:** Pearson r, AUROC, AUPRC, EF@25% (computed from TSV — not hardcoded)  
#@markdown
#@markdown > **해석:** CNN pose score는 top-pose PASS/FAIL 전역 분류기로 강합니다 (AUROC 0.958).  
#@markdown > Within-case pose ranking에서 오류 발생 가능 — 이것이 12/16 vs 15/16 gap의 원인.  
#@markdown > best-of-9 Pearson은 탐색 능력 진단 지표이며 운영 지표가 아닙니다.

import pandas as pd
import numpy as np
from scipy import stats
from sklearn.metrics import roc_auc_score, average_precision_score

RCSB_TSV = DAD_ROOT / '06_Report' / 'Mr_Repro' / 'external_validation' / 'rcsb_seed' / 'redocking_results.tsv'

if RCSB_TSV.exists():
    df_bench = pd.read_csv(RCSB_TSV, sep='\t')
    print(f'Loaded RCSB seed benchmark: {len(df_bench)} cases')
    print(f'Families: {df_bench["family"].unique().tolist()}')

    valid = df_bench.dropna(subset=['cnn_pose_score', 'top_pose_rmsd_a'])
    pearson_r, p_val = stats.pearsonr(valid['cnn_pose_score'], valid['top_pose_rmsd_a'])
    spearman_r, _ = stats.spearmanr(valid['cnn_pose_score'], valid['top_pose_rmsd_a'])
    print(f'\nCNN pose vs top RMSD:')
    print(f'  Pearson r  = {pearson_r:.3f}  (p={p_val:.4f})')
    print(f'  Spearman r = {spearman_r:.3f}')

    valid['pass_label'] = (valid['top_pose_rmsd_a'] < 2.0).astype(int)
    if valid['pass_label'].nunique() == 2:
        auroc = roc_auc_score(valid['pass_label'], valid['cnn_pose_score'])
        auprc = average_precision_score(valid['pass_label'], valid['cnn_pose_score'])
        n25   = max(1, int(len(valid) * 0.25))
        top25 = valid.nlargest(n25, 'cnn_pose_score')
        ef25  = (top25['pass_label'].sum() / n25) / valid['pass_label'].mean()
        print(f'\nCNN as PASS/FAIL classifier:')
        print(f'  AUROC  = {auroc:.3f}')
        print(f'  AUPRC  = {auprc:.3f}')
        print(f'  EF@25% = {ef25:.3f}')

    valid2 = df_bench.dropna(subset=['cnn_pose_score', 'best_pose_rmsd_a'])
    if len(valid2) > 2:
        pearson_b9, _ = stats.pearsonr(valid2['cnn_pose_score'], valid2['best_pose_rmsd_a'])
        print(f'\nCNN pose vs best-of-9 RMSD (diagnostic only):')
        print(f'  Pearson r = {pearson_b9:.3f}  (weakened vs top-RMSD — expected)')
        print('  NOTE: best-of-9 = search-recovery diagnostic, not operational metric.')

    print('\nFamily coverage:')
    summary = df_bench.groupby('family').agg(
        n_cases=('case_id', 'count'),
        pass_rate=('redock_pass', lambda x: (x == 'PASS').mean()),
        mean_rmsd=('top_pose_rmsd_a', 'mean'),
    ).round(3)
    print(summary.to_string())

else:
    print(f'RCSB TSV not found at {RCSB_TSV}')
    print('Set MOUNT_DRIVE=True and DRIVE_DAD_ROOT to your Drive DAD folder,')
    print('or run on a local machine with access to the RCSB seed data.')
    df_bench = None

In [ ]:
#@title 8. Structure Visualization — py3Dmol (Stage 10) {display-mode: "form"}
#@markdown **예상 실행 시간 / Expected time:** < 1 분  
#@markdown **Expected output:** interactive 3D viewer (live mode only; skipped in replay)  
#@markdown > Replay 모드에서는 SDF 파일이 없으므로 시각화를 건너뜁니다.

try:
    import py3Dmol
    PY3DMOL_AVAILABLE = True
except ImportError:
    PY3DMOL_AVAILABLE = False
    print('py3Dmol not available')


def visualize_docking(result: DockingResult, struct_paths: dict):
    if result.output_sdf == '<replay:no_sdf>':
        print(f'  {result.case_id}: visualization skipped (replay mode — no SDF)')
        return
    if not PY3DMOL_AVAILABLE:
        return
    prot_pdb = struct_paths.get(result.protein_id)
    sdf_path = result.output_sdf
    if not (prot_pdb and Path(str(prot_pdb)).exists() and Path(sdf_path).exists()):
        print(f'  {result.case_id}: missing PDB or SDF')
        return
    view = py3Dmol.view(width=800, height=500)
    view.addModel(Path(str(prot_pdb)).read_text(), 'pdb')
    view.setStyle({'chain': 'A'}, {'cartoon': {'color': 'lightblue'}})
    view.addModel(Path(sdf_path).read_text(), 'sdf')
    view.setStyle({'model': -1}, {'stick': {'colorscheme': 'greenCarbon'}})
    view.zoomTo({'model': -1})
    view.show()
    print(f'  {result.case_id}: top-pose RMSD = {result.top_pose_rmsd} Å')


if docking_results:
    for r in docking_results[:3]:  # show first 3
        visualize_docking(r, structure_paths)
else:
    print('No docking results to visualize.')

In [ ]:
#@title 9. Result Aggregation (Stage 11) {display-mode: "form"}
#@markdown **예상 실행 시간 / Expected time:** < 1 분  
#@markdown **Expected output:** `docking_results.tsv` + top-5 CNN pose table

import pandas as pd

rows = [{
    'case_id':          r.case_id,
    'protein_id':       r.protein_id,
    'ligand_id':        r.ligand_id,
    'vina_score':       r.vina_score,
    'cnn_pose_score':   r.cnn_pose_score,
    'cnn_affinity':     r.cnn_affinity,
    'top_pose_rmsd_a':  r.top_pose_rmsd,
    'best_pose_rmsd_a': r.best_pose_rmsd,
    'pose_count':       r.pose_count,
    'status':           r.status,
    'gnina_version':    r.gnina_version,
    'runtime_s':        r.runtime_s,
} for r in docking_results]

df_results = pd.DataFrame(rows)

out_tsv = WORK_DIR / 'results' / 'docking_results.tsv'
df_results.to_csv(out_tsv, sep='\t', index=False)
print(f'Results saved: {out_tsv}')

n_pass  = (df_results['status'].isin(['PASS', 'replay'])).sum()
n_total = len(df_results)
print(f'\nSummary: {n_pass}/{n_total} cases')

if not df_results.empty:
    print('\nTop cases by CNN pose score:')
    top = df_results.nlargest(min(5, len(df_results)), 'cnn_pose_score')
    for _, row in top.iterrows():
        rmsd_str = f"{row['top_pose_rmsd_a']:.3f} Å" if pd.notna(row['top_pose_rmsd_a']) else 'N/A'
        print(f"  {row['case_id']}: cnn_pose={row['cnn_pose_score']:.4f}, "
              f"vina={row['vina_score']:.2f}, RMSD={rmsd_str}")

In [ ]:
#@title 10. Manuscript Tables (Table 5 & 6) {display-mode: "form"}
#@markdown **예상 실행 시간 / Expected time:** < 1 분  
#@markdown **Expected output:** table5_family_coverage.tsv + table6_failure_cases.tsv

import pandas as pd
import numpy as np

TABLES_DIR = WORK_DIR / 'results' / 'tables'
TABLES_DIR.mkdir(parents=True, exist_ok=True)

if df_bench is not None and len(df_bench) > 0:
    # Table 5: Family Coverage
    t5 = df_bench.groupby('family').agg(
        N=('case_id', 'count'),
        pass_top=('redock_pass', lambda x: (x == 'PASS').sum()),
        mean_top_rmsd=('top_pose_rmsd_a', 'mean'),
        median_top_rmsd=('top_pose_rmsd_a', 'median'),
        mean_cnn_pose=('cnn_pose_score', 'mean'),
    ).reset_index()
    t5['pass_rate_pct'] = (t5['pass_top'] / t5['N'] * 100).round(1)
    t5 = t5.round(3)
    print('=== Table 5: Family Coverage ===')
    print(t5.to_string(index=False))
    t5.to_csv(TABLES_DIR / 'table5_family_coverage.tsv', sep='\t', index=False)

    # Table 6: Failure Cases
    failures = df_bench[df_bench['redock_pass'] == 'FAIL'].copy()
    if len(failures) > 0:
        def classify_failure(row):
            top  = row.get('top_pose_rmsd_a', float('inf'))
            best = row.get('best_pose_rmsd_a', float('inf'))
            if pd.isna(top): return 'no_pose'
            if best < 2.0 and top >= 2.0: return 'ranking_error'
            return 'search_failure'
        failures['failure_type'] = failures.apply(classify_failure, axis=1)
        avail = [c for c in ['case_id','family','top_pose_rmsd_a','best_pose_rmsd_a',
                              'cnn_pose_score','failure_type'] if c in failures.columns]
        print('\n=== Table 6: Failure Cases ===')
        print(failures[avail].to_string(index=False))
        failures[avail].to_csv(TABLES_DIR / 'table6_failure_cases.tsv', sep='\t', index=False)
        print('\nFailure types:', failures['failure_type'].value_counts().to_dict())
    else:
        print('No failure cases.')
else:
    print('RCSB benchmark data not available — skipping manuscript tables.')
    if df_results is not None and len(df_results) > 0:
        print('Tier 1 docking results:')
        print(df_results.to_string(index=False))

In [ ]:
#@title 11. Manuscript Figures (PNG + PDF, 300 dpi) {display-mode: "form"}
#@markdown **예상 실행 시간 / Expected time:** < 2 분  
#@markdown **Expected output:** fig1 (RMSD boxplot), fig2 (CNN scatter), fig3 (ROC) — PNG + PDF  
#@markdown
#@markdown **Note:** Pearson r, AUROC, EF@25% are computed from TSV — not hardcoded.  
#@markdown best-of-9 is labeled "search-recovery diagnostic" in all figures.

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from scipy import stats
from sklearn.metrics import roc_curve, auc

FIGS_DIR = WORK_DIR / 'results' / 'figures'
FIGS_DIR.mkdir(parents=True, exist_ok=True)

FAMILY_LABELS = {
    'crp_fnr_regulator':         'CRP/FNR',
    'mcp_ligand_binding_domain':  'MCP LBD',
    'amino_acid_binding':         'AA Binding',
    'periplasmic_sugar_binding':   'Sugar Binding',
}
FAMILY_COLORS = {
    'crp_fnr_regulator':         '#2196F3',
    'mcp_ligand_binding_domain':  '#4CAF50',
    'amino_acid_binding':         '#FF9800',
    'periplasmic_sugar_binding':   '#9C27B0',
}


def save_fig(fig, stem):
    for ext in ['png', 'pdf']:
        fig.savefig(str(FIGS_DIR / f'{stem}.{ext}'), dpi=300, bbox_inches='tight')
    print(f'Saved: {stem}.{{png,pdf}}')


if df_bench is not None and len(df_bench) > 0:
    families = df_bench['family'].unique()

    # Figure 1: RMSD Boxplot
    fig, ax = plt.subplots(figsize=(10, 5))
    positions, labels, data_top, data_best = [], [], [], []
    for i, fam in enumerate(families):
        sub = df_bench[df_bench['family'] == fam]
        data_top.append(sub['top_pose_rmsd_a'].dropna().tolist())
        data_best.append(sub['best_pose_rmsd_a'].dropna().tolist())
        labels.append(FAMILY_LABELS.get(fam, fam))
        positions.append(i * 3)
    w = 0.8
    ax.boxplot(data_top,  positions=positions,           widths=w, patch_artist=True,
               boxprops=dict(facecolor='#2196F3', alpha=0.7))
    ax.boxplot(data_best, positions=[p + 1 for p in positions], widths=w, patch_artist=True,
               boxprops=dict(facecolor='#FF9800', alpha=0.7))
    ax.axhline(2.0, color='red', linestyle='--', linewidth=1.2)
    ax.set_xticks([p + 0.5 for p in positions])
    ax.set_xticklabels(labels, fontsize=10)
    ax.set_ylabel('RMSD (Å)', fontsize=12)
    ax.set_title('Top-pose vs. Best-of-9 RMSD by Family\n'
                 '(best-of-9 = search-recovery diagnostic only)', fontsize=11)
    ax.legend(handles=[
        mpatches.Patch(color='#2196F3', alpha=0.7, label='Top pose (operational)'),
        mpatches.Patch(color='#FF9800', alpha=0.7, label='Best-of-9 (diagnostic)'),
        plt.Line2D([0], [0], color='red', linestyle='--', label='2 Å threshold'),
    ], loc='upper right', fontsize=9)
    plt.tight_layout()
    save_fig(fig, 'fig1_rmsd_boxplot')
    plt.show()

    # Figure 2: CNN vs RMSD scatter
    valid = df_bench.dropna(subset=['cnn_pose_score', 'top_pose_rmsd_a'])
    pearson_r, p_val = stats.pearsonr(valid['cnn_pose_score'], valid['top_pose_rmsd_a'])
    fig, ax = plt.subplots(figsize=(7, 5))
    for fam in families:
        sub = valid[valid['family'] == fam]
        ax.scatter(sub['cnn_pose_score'], sub['top_pose_rmsd_a'],
                   label=FAMILY_LABELS.get(fam, fam), color=FAMILY_COLORS.get(fam, 'grey'),
                   s=70, zorder=3, edgecolors='white', linewidths=0.5)
    x_fit = np.linspace(valid['cnn_pose_score'].min(), valid['cnn_pose_score'].max(), 100)
    slope, intercept, *_ = stats.linregress(valid['cnn_pose_score'], valid['top_pose_rmsd_a'])
    ax.plot(x_fit, slope * x_fit + intercept, 'k--', linewidth=1, alpha=0.6)
    ax.axhline(2.0, color='red', linestyle=':', linewidth=1.2, alpha=0.7)
    ax.set_xlabel('CNN Pose Score', fontsize=12)
    ax.set_ylabel('Top-pose RMSD (Å)', fontsize=12)
    ax.set_title(f'CNN Score vs Top-pose RMSD\n'
                 f'Pearson r = {pearson_r:.3f}  (p = {p_val:.4f}, n={len(valid)})', fontsize=11)
    ax.legend(fontsize=9, loc='upper left')
    plt.tight_layout()
    save_fig(fig, 'fig2_cnn_rmsd_scatter')
    plt.show()

    # Figure 3: ROC Curve
    valid['pass_label'] = (valid['top_pose_rmsd_a'] < 2.0).astype(int)
    if valid['pass_label'].nunique() == 2:
        fpr, tpr, _ = roc_curve(valid['pass_label'], valid['cnn_pose_score'])
        auroc_val = auc(fpr, tpr)
        fig, ax = plt.subplots(figsize=(5.5, 5))
        ax.plot(fpr, tpr, color='#2196F3', linewidth=2,
                label=f'CNN pose score (AUROC = {auroc_val:.3f})')
        ax.plot([0, 1], [0, 1], 'k--', linewidth=1, alpha=0.5, label='Random')
        ax.fill_between(fpr, tpr, alpha=0.15, color='#2196F3')
        ax.set_xlabel('False Positive Rate', fontsize=12)
        ax.set_ylabel('True Positive Rate', fontsize=12)
        ax.set_title('ROC Curve: CNN Score as\nTop-pose PASS/FAIL Classifier', fontsize=11)
        ax.legend(fontsize=10, loc='lower right')
        plt.tight_layout()
        save_fig(fig, 'fig3_roc_curve')
        plt.show()

    print('\nAll manuscript figures generated.')

    # Download figures to local machine (Colab)
    if IN_COLAB:
        from google.colab import files
        import zipfile
        zip_path = WORK_DIR / 'results' / 'figures.zip'
        with zipfile.ZipFile(str(zip_path), 'w') as zf:
            for f in FIGS_DIR.glob('*'):
                zf.write(str(f), f.name)
        files.download(str(zip_path))
        print('figures.zip download started.')
else:
    print('RCSB benchmark data not available — skipping manuscript figures.')
    print('Set MOUNT_DRIVE=True and DRIVE_DAD_ROOT to your Drive DAD folder.')

In [ ]:
#@title 12. Reproducibility Footprint + Download {display-mode: "form"}
#@markdown **예상 실행 시간 / Expected time:** < 1 분  
#@markdown **Expected output:** `reproducibility_footprint.json` + auto-download in Colab  
#@markdown Use this JSON for the Methods section: Python version, GNINA version, package versions, GPU info.

import platform, sys, subprocess, json
from datetime import datetime, timezone

footprint = {
    'recorded_at': datetime.now(timezone.utc).isoformat(),
    'python':      sys.version,
    'platform':    platform.platform(),
    'exec_mode':   EXEC_MODE,
    'in_colab':    IN_COLAB,
    'n_cases_run': len(docking_results),
    'dad_root':    str(DAD_ROOT),
}

for pkg in ['rdkit', 'pandas', 'numpy', 'matplotlib', 'scipy', 'scikit-learn', 'biopython']:
    try:
        import importlib.metadata
        footprint[f'pkg_{pkg}'] = importlib.metadata.version(pkg)
    except Exception:
        footprint[f'pkg_{pkg}'] = 'unknown'

GNINA_BIN = Path('/usr/local/bin/gnina')
try:
    r = subprocess.run([str(GNINA_BIN), '--version'], capture_output=True, text=True, timeout=5)
    footprint['gnina_version'] = (r.stdout + r.stderr).strip()[:100]
except Exception:
    footprint['gnina_version'] = 'not available'

try:
    r = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version',
                        '--format=csv,noheader'], capture_output=True, text=True, timeout=5)
    footprint['gpu'] = r.stdout.strip()
except Exception:
    footprint['gpu'] = 'not available'

fp_path = WORK_DIR / 'results' / 'reproducibility_footprint.json'
fp_path.write_text(json.dumps(footprint, indent=2))
print(f'Footprint saved: {fp_path}')

print('\n=== Reproducibility Summary ===')
print(f'  Execution mode : {EXEC_MODE}')
print(f'  Runtime        : {"Google Colab" if IN_COLAB else "Local/WSL2"}')
print(f'  GPU            : {footprint.get("gpu", "N/A")}')
print(f'  GNINA          : {footprint.get("gnina_version", "N/A")[:60]}')
print(f'  Python         : {sys.version.split()[0]}')
print(f'  Cases run      : {footprint["n_cases_run"]}')
print(f'  DAD root       : {footprint["dad_root"]}')
print()
print('Tolerance thresholds (comparable across runtimes):')
print('  Vina score   : ±0.5 kcal/mol')
print('  CNN pose     : ±0.05')
print('  CNN affinity : ±0.5 pKi')
print()
print('Validation status: 16-case RCSB seed benchmark (12/16 top-pose RMSD < 2 Å)')
print('Presubmission-enquiry candidate — not yet a full Nature Protocols submission.')

# Download footprint JSON (Colab)
if IN_COLAB:
    from google.colab import files
    files.download(str(fp_path))
    print('reproducibility_footprint.json download started.')